# M12a · Espaces vectoriels et applications linéaires — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m12a_espaces_vectoriels/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m12a_espaces_vectoriels/03_notebook.ipynb`, créée par `python tools/start_chapter.py M12a`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M12a.1 à M12a.13, M12a.20, M12a.21 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M12a.26 | Tester l'indépendance et extraire une base de l'espace des colonnes · `is_independent, column_space_basis` | 🔨 | ★★ | 20 |
|  | M12a.27 | Une base du noyau, colonne libre par colonne libre · `null_space_basis` | 🔨 | ★★ | 25 |
|  | M12a.28 | Coordonnées dans une base et changement de base · `coordinates, change_of_basis` | 🔨 | ★ | 20 |
|  | M12a.29 | `sympy.Matrix.nullspace`, `columnspace` et `rank` contre tes fonctions | 📦 | ★ | 15 |
| B | M12a.30 | Prédire le rang : matrices aléatoires, produits, colonnes répétées | 🔮 | ★ | 10 |
|  | M12a.31 | Le rang numérique : une colonne presque redondante et le choix de la tolérance | 🔬 | ★★ | 20 |
|  | M12a.32 | Une application linéaire du plan : l'image du carré unité, et ce qu'elle écrase quand son noyau n'est pas nul | 🎨 | ★ | 20 |
| C | M12a.33 | Chasse à l'erreur : une « base » liée, un noyau faux, S⁻¹AS écrit à l'envers | 🐛 | ★ | 15 |
|  | M12a.34 | Un test de propriété : A fois une base du noyau donne 0, et le théorème du rang sur des matrices aléatoires | 🛠️ | ★ | 15 |
|  | M12a.35 | Défi : retrouver les relations linéaires exactes cachées entre les colonnes d'un tableau, sans en inventer | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : sous-espaces, combinaisons et indépendance, bases et coordonnées, rangs, matrices d'applications et composées, noyaux, images et solutions d'un système, changement de base, application affine et sa réciproque, dérivation des polynômes, une feature redondante, un encodeur qui écrase une direction.
- Écrire les cinq fonctions de `mymath/linmaps.py` et les tester, puis comparer avec SymPy et NumPy.
- Prédire des rangs, mesurer le rang numérique, dessiner ce qu'écrase une application, chasser des erreurs, tester des propriétés, retrouver des relations cachées.

**Rappel express.** Sous-espace : $\mathbf{0}$ dedans, stable par $\lambda\mathbf{x} + \mu\mathbf{y}$ ; libre : chaque colonne de $[\mathbf{v}_1 \cdots \mathbf{v}_k]$ porte un pivot ; coordonnées : $\mathbf{B}\mathbf{c} = \mathbf{v}$ ; $\operatorname{rg}\mathbf{A} = \operatorname{rg}\mathbf{A}^\top$, $\operatorname{rg}(\mathbf{A}\mathbf{B}) \le \min(\operatorname{rg}\mathbf{A}, \operatorname{rg}\mathbf{B})$ ; colonne $j$ de la matrice de $f$ = $f(\mathbf{e}_j)$ ; « $f$ puis $g$ » = $\mathbf{B}\mathbf{A}$ ; $\dim\ker\mathbf{A} + \operatorname{rg}\mathbf{A} = n$ ; solutions de $\mathbf{A}\mathbf{x} = \mathbf{b}$ = $\mathbf{x}_p + \ker\mathbf{A}$ ; nouvelle base : $\mathbf{S}^{-1}\mathbf{A}\mathbf{S}$ ; $\varphi^{-1}(\mathbf{y}) = \mathbf{A}^{-1}(\mathbf{y} - \mathbf{b})$.

## Partie 0 · Vérifier tes exercices papier (✏️ M12a.1 à M12a.13, 🔗 M12a.20 et M12a.21)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0, -2, 5]` pour un vecteur, des numéros de colonnes ou une forme, une liste de lignes `[[1, 0], [-2, 3]]` pour une matrice, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

**Ex M12a.1 — Reconnaître des sous-espaces de ℝ³ et de l'espace des polynômes**

In [ ]:
# Ex M12a.1: the values you found on paper
answer_M12a_1a = ...  # a) True or False
answer_M12a_1b = ...  # b) True or False
answer_M12a_1c = ...  # c) True or False
answer_M12a_1d = ...  # d) True or False
answer_M12a_1e = ...  # e) True or False
answer_M12a_1f = ...  # f) True or False
answer_M12a_1g = ...  # g) True or False
answer_M12a_1h = ...  # h) a list of 3 numbers

for letter, answer in zip("abcdefgh", [answer_M12a_1a, answer_M12a_1b, answer_M12a_1c, answer_M12a_1d, answer_M12a_1e, answer_M12a_1f, answer_M12a_1g, answer_M12a_1h]):
    wbm.check(f"M12a.1{letter}", answer)

**Ex M12a.2 — Écrire un vecteur comme combinaison linéaire : un système à résoudre**

In [ ]:
# Ex M12a.2: the values you found on paper
answer_M12a_2a = ...  # a) a list of 2 numbers
answer_M12a_2b = ...  # b) True or False
answer_M12a_2c = ...  # c) a number
answer_M12a_2d = ...  # d) a list of 3 numbers
answer_M12a_2e = ...  # e) True or False
answer_M12a_2f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M12a_2a, answer_M12a_2b, answer_M12a_2c, answer_M12a_2d, answer_M12a_2e, answer_M12a_2f]):
    wbm.check(f"M12a.2{letter}", answer)

**Ex M12a.3 — L'indépendance de trois vecteurs de ℝ³ par le pivot de Gauss**

In [ ]:
# Ex M12a.3: the values you found on paper
answer_M12a_3a = ...  # a) an integer
answer_M12a_3b = ...  # b) True or False
answer_M12a_3c = ...  # c) a list of 2 numbers
answer_M12a_3d = ...  # d) a list of 3 numbers
answer_M12a_3e = ...  # e) a number
answer_M12a_3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_M12a_3a, answer_M12a_3b, answer_M12a_3c, answer_M12a_3d, answer_M12a_3e, answer_M12a_3f]):
    wbm.check(f"M12a.3{letter}", answer)

**Ex M12a.4 — Extraire une base d'une famille génératrice, compléter une famille libre**

In [ ]:
# Ex M12a.4: the values you found on paper
answer_M12a_4a = ...  # a) an integer
answer_M12a_4b = ...  # b) a list of integers
answer_M12a_4c = ...  # c) a list of 2 numbers
answer_M12a_4d = ...  # d) an integer
answer_M12a_4e = ...  # e) a list of 2 numbers
answer_M12a_4f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M12a_4a, answer_M12a_4b, answer_M12a_4c, answer_M12a_4d, answer_M12a_4e, answer_M12a_4f]):
    wbm.check(f"M12a.4{letter}", answer)

**Ex M12a.5 — Les coordonnées d'un vecteur dans une base qui n'est pas la base canonique**

In [ ]:
# Ex M12a.5: the values you found on paper
answer_M12a_5a = ...  # a) a list of 3 numbers
answer_M12a_5b = ...  # b) a list of 3 numbers
answer_M12a_5c = ...  # c) a list of 3 numbers (1 decimal)
answer_M12a_5d = ...  # d) a list of 2 numbers
answer_M12a_5e = ...  # e) True or False
answer_M12a_5f = ...  # f) a list of 3 numbers

for letter, answer in zip("abcdef", [answer_M12a_5a, answer_M12a_5b, answer_M12a_5c, answer_M12a_5d, answer_M12a_5e, answer_M12a_5f]):
    wbm.check(f"M12a.5{letter}", answer)

**Ex M12a.6 — Le rang par les colonnes et par les lignes, celui d'un produit, et celui d'un tableau centré**

In [ ]:
# Ex M12a.6: the values you found on paper
answer_M12a_6a = ...  # a) an integer
answer_M12a_6b = ...  # b) an integer
answer_M12a_6c = ...  # c) an integer
answer_M12a_6d = ...  # d) an integer
answer_M12a_6e = ...  # e) an integer
answer_M12a_6f = ...  # f) an integer
answer_M12a_6g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M12a_6a, answer_M12a_6b, answer_M12a_6c, answer_M12a_6d, answer_M12a_6e, answer_M12a_6f, answer_M12a_6g]):
    wbm.check(f"M12a.6{letter}", answer)

**Ex M12a.7 — La matrice d'une application linéaire : les images des vecteurs de base**

In [ ]:
# Ex M12a.7: the values you found on paper
answer_M12a_7a = ...  # a) a matrix
answer_M12a_7b = ...  # b) a list of 2 numbers
answer_M12a_7c = ...  # c) a list of 2 numbers
answer_M12a_7d = ...  # d) a list of 2 integers
answer_M12a_7e = ...  # e) a matrix
answer_M12a_7f = ...  # f) a matrix (1 decimal)
answer_M12a_7g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M12a_7a, answer_M12a_7b, answer_M12a_7c, answer_M12a_7d, answer_M12a_7e, answer_M12a_7f, answer_M12a_7g]):
    wbm.check(f"M12a.7{letter}", answer)

**Ex M12a.8 — Composer deux applications du plan, c'est multiplier leurs matrices**

In [ ]:
# Ex M12a.8: the values you found on paper
answer_M12a_8a = ...  # a) a matrix
answer_M12a_8b = ...  # b) a matrix
answer_M12a_8c = ...  # c) a list of 2 numbers
answer_M12a_8d = ...  # d) True or False
answer_M12a_8e = ...  # e) a matrix
answer_M12a_8f = ...  # f) a matrix
answer_M12a_8g = ...  # g) a matrix (4 decimals)

for letter, answer in zip("abcdefg", [answer_M12a_8a, answer_M12a_8b, answer_M12a_8c, answer_M12a_8d, answer_M12a_8e, answer_M12a_8f, answer_M12a_8g]):
    wbm.check(f"M12a.8{letter}", answer)

**Ex M12a.9 — Noyau et image d'une matrice 3 × 4, et le théorème du rang**

In [ ]:
# Ex M12a.9: the values you found on paper
answer_M12a_9a = ...  # a) an integer
answer_M12a_9b = ...  # b) a list of integers
answer_M12a_9c = ...  # c) an integer
answer_M12a_9d = ...  # d) a list of 4 numbers
answer_M12a_9e = ...  # e) a list of 4 numbers
answer_M12a_9f = ...  # f) a matrix
answer_M12a_9g = ...  # g) a list of 3 numbers
answer_M12a_9h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12a_9a, answer_M12a_9b, answer_M12a_9c, answer_M12a_9d, answer_M12a_9e, answer_M12a_9f, answer_M12a_9g, answer_M12a_9h]):
    wbm.check(f"M12a.9{letter}", answer)

**Ex M12a.10 — Toutes les solutions de A x = b : une solution particulière plus le noyau**

In [ ]:
# Ex M12a.10: the values you found on paper
answer_M12a_10a = ...  # a) a list of 4 numbers
answer_M12a_10b = ...  # b) an integer
answer_M12a_10c = ...  # c) a list of 4 numbers
answer_M12a_10d = ...  # d) a list of 4 numbers
answer_M12a_10e = ...  # e) a list of 4 numbers
answer_M12a_10f = ...  # f) True or False
answer_M12a_10g = ...  # g) a number
answer_M12a_10h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M12a_10a, answer_M12a_10b, answer_M12a_10c, answer_M12a_10d, answer_M12a_10e, answer_M12a_10f, answer_M12a_10g, answer_M12a_10h]):
    wbm.check(f"M12a.10{letter}", answer)

**Ex M12a.11 — Une base adaptée qui rend diagonale la matrice d'une application : S⁻¹AS**

In [ ]:
# Ex M12a.11: the values you found on paper
answer_M12a_11a = ...  # a) a list of 2 numbers
answer_M12a_11b = ...  # b) a list of 2 numbers
answer_M12a_11c = ...  # c) a matrix
answer_M12a_11d = ...  # d) a matrix
answer_M12a_11e = ...  # e) a list of 2 numbers
answer_M12a_11f = ...  # f) a list of 2 numbers
answer_M12a_11g = ...  # g) a list of 2 numbers
answer_M12a_11h = ...  # h) a matrix

for letter, answer in zip("abcdefgh", [answer_M12a_11a, answer_M12a_11b, answer_M12a_11c, answer_M12a_11d, answer_M12a_11e, answer_M12a_11f, answer_M12a_11g, answer_M12a_11h]):
    wbm.check(f"M12a.11{letter}", answer)

**Ex M12a.12 — Une application affine du plan : partie linéaire, translation, inverse**

In [ ]:
# Ex M12a.12: the values you found on paper
answer_M12a_12a = ...  # a) a matrix
answer_M12a_12b = ...  # b) a list of 2 numbers
answer_M12a_12c = ...  # c) a list of 2 numbers
answer_M12a_12d = ...  # d) a matrix
answer_M12a_12e = ...  # e) a list of 2 numbers
answer_M12a_12f = ...  # f) a list of 2 numbers
answer_M12a_12g = ...  # g) a list of 2 numbers
answer_M12a_12h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12a_12a, answer_M12a_12b, answer_M12a_12c, answer_M12a_12d, answer_M12a_12e, answer_M12a_12f, answer_M12a_12g, answer_M12a_12h]):
    wbm.check(f"M12a.12{letter}", answer)

**Ex M12a.13 — La dérivation sur les polynômes de degré au plus 3 : matrice, noyau, image**

In [ ]:
# Ex M12a.13: the values you found on paper
answer_M12a_13a = ...  # a) a matrix
answer_M12a_13b = ...  # b) a list of 4 numbers
answer_M12a_13c = ...  # c) a list of 4 numbers
answer_M12a_13d = ...  # d) an integer
answer_M12a_13e = ...  # e) an integer
answer_M12a_13f = ...  # f) True or False
answer_M12a_13g = ...  # g) a matrix
answer_M12a_13h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M12a_13a, answer_M12a_13b, answer_M12a_13c, answer_M12a_13d, answer_M12a_13e, answer_M12a_13f, answer_M12a_13g, answer_M12a_13h]):
    wbm.check(f"M12a.13{letter}", answer)

**Ex M12a.20 — Pont : une feature somme de deux autres rend XᵀX singulière, et le noyau dit laquelle**

In [ ]:
# Ex M12a.20: the values you found on paper
answer_M12a_20a = ...  # a) an integer
answer_M12a_20b = ...  # b) an integer
answer_M12a_20c = ...  # c) a list of 3 numbers
answer_M12a_20d = ...  # d) an integer
answer_M12a_20e = ...  # e) a list of 4 numbers (1 decimal)
answer_M12a_20f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_M12a_20a, answer_M12a_20b, answer_M12a_20c, answer_M12a_20d, answer_M12a_20e, answer_M12a_20f]):
    wbm.check(f"M12a.20{letter}", answer)

**Ex M12a.21 — Pont : projeter sur un plan puis revenir, ce que la transformation inverse ne peut pas rendre**

In [ ]:
# Ex M12a.21: the values you found on paper
answer_M12a_21a = ...  # a) an integer
answer_M12a_21b = ...  # b) a list of 3 numbers
answer_M12a_21c = ...  # c) a list of 2 numbers
answer_M12a_21d = ...  # d) True or False
answer_M12a_21e = ...  # e) a matrix
answer_M12a_21f = ...  # f) a list of 3 numbers
answer_M12a_21g = ...  # g) a list of 3 numbers

for letter, answer in zip("abcdefg", [answer_M12a_21a, answer_M12a_21b, answer_M12a_21c, answer_M12a_21d, answer_M12a_21e, answer_M12a_21f, answer_M12a_21g]):
    wbm.check(f"M12a.21{letter}", answer)

## Partie A · Ta librairie : indépendance, image, noyau, coordonnées, changement de base ; SymPy et NumPy

Fiche §2 à §7. Les cinq fonctions de `mymath/linmaps.py`, puis les mêmes calculs avec SymPy et NumPy.

In [ ]:
# Tools of the notebook (parts A to C)
import matplotlib.pyplot as plt
import numpy as np
import sympy

from wbm.nbtools import StopCheck, checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M12a.26 — Tester l'indépendance et extraire une base de l'espace des colonnes · `is_independent, column_space_basis` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer le test d'indépendance d'une famille et l'extraction d'une base de l'espace des colonnes, par les colonnes à pivot.  
**Prérequis :** fiche §2 à §4 ; M09a §6 (`rref`) · **mymath :** `linmaps.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/linmaps.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Une famille de vecteurs est donnée par les **colonnes** d'une matrice (une liste de lignes ou un tableau NumPy à deux dimensions). NumPy est permis, `@`, `.T` et `np.linalg.solve` compris ; `np.linalg.matrix_rank`, `svd`, `qr`, `lstsq`, `pinv`, `inv`, `scipy.linalg` et SymPy sont interdits : les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas. Les pivots viennent de la forme échelonnée réduite calculée avec pivot partiel et la tolérance `tol` : tu peux recopier ta fonction `rref` de M09a dans une fonction privée `_rref` du fichier. Ne modifie jamais la matrice reçue : travaille sur une copie, `np.array(A, dtype=float)`.

Écris (fiche §2 à §4), en suivant les docstrings :
- `is_independent(A, tol=1e-10)` : `True` si les colonnes de $\mathbf{A}$ sont libres, c'est-à-dire si **chaque** colonne porte un pivot ; un `bool` de Python ;
- `column_space_basis(A, tol=1e-10)` : les colonnes **de $\mathbf{A}$** (pas de la forme réduite) dont l'indice porte un pivot, dans l'ordre, en tableau de floats de forme $(m, r)$ ; la forme $(m, 0)$ si $\mathbf{A}$ est nulle.

Les deux lèvent une `ValueError` si $\mathbf{A}$ n'est pas à deux dimensions. Une fonction privée `_rref(A, tol)` qui rend la forme réduite et les colonnes des pivots sert aux deux (et à M12a.27).

La vérification calcule :
- a) `is_independent([[2, 1, 0], [1, -1, 3], [0, 2, 1]])` ;
- b) `is_independent([[1, 2, 4], [0, 1, 1], [3, 1, 7], [2, -1, 3]])` ;
- c) `is_independent([[1, 0, 2], [0, 1, 5]])` ;
- d) `column_space_basis([[1, 2, 4], [0, 1, 1], [3, 1, 7], [2, -1, 3]])` (une matrice) ;
- e) `column_space_basis([[0, 0, 1], [0, 0, 2]])` (une matrice) ;
- f) le nom de l'erreur levée par `is_independent([1, 2])` ;
- g) vrai ou faux : `is_independent` rend un `bool` de Python (`type(...) is bool`).

Dans ta copie : en b), quelle colonne combine les deux autres, et avec quels coefficients ? Pourquoi c) est-il liée sans calcul ? Pourquoi, en e), la première colonne est-elle absente du résultat ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")  # reload your saved file

In [ ]:
with checked("M12a.26a"):
    wbm.check("M12a.26a", mymath.linmaps.is_independent([[2, 1, 0], [1, -1, 3], [0, 2, 1]]), computed=True)
with checked("M12a.26b"):
    wbm.check("M12a.26b", mymath.linmaps.is_independent([[1, 2, 4], [0, 1, 1], [3, 1, 7], [2, -1, 3]]), computed=True)
with checked("M12a.26c"):
    wbm.check("M12a.26c", mymath.linmaps.is_independent([[1, 0, 2], [0, 1, 5]]), computed=True)
with checked("M12a.26d"):
    wbm.check("M12a.26d", np.asarray(mymath.linmaps.column_space_basis(
        [[1, 2, 4], [0, 1, 1], [3, 1, 7], [2, -1, 3]])).tolist(), computed=True)
with checked("M12a.26e"):
    wbm.check("M12a.26e", np.asarray(mymath.linmaps.column_space_basis([[0, 0, 1], [0, 0, 2]])).tolist(),
              computed=True)
with checked("M12a.26f"):
    wbm.check("M12a.26f", raised(mymath.linmaps.is_independent, [1, 2]), computed=True)
with checked("M12a.26g"):
    wbm.check("M12a.26g", type(mymath.linmaps.is_independent([[1, 0], [0, 1]])) is bool, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12a_linmaps.py", "test_is_independent_ or test_column_space_basis_ or test_linmaps_uses_")

### Ex M12a.27 — Une base du noyau, colonne libre par colonne libre · `null_space_basis` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer une base du noyau à partir de la forme échelonnée réduite, avec la convention de SymPy.  
**Prérequis :** fiche §6 ; Ex M12a.26 · **mymath :** `linmaps.py` · **Parcours :** C, E

Écris `null_space_basis(A, tol=1e-10)` (fiche §6), en suivant la docstring : réduis $\mathbf{A}$ (ta fonction `_rref`), repère les colonnes **libres** (sans pivot), et, pour chacune, dans l'ordre, construis un vecteur $\mathbf{x}$ de $\mathbb{R}^n$ : $x_j = 1$ pour cette colonne libre $j$, 0 pour les autres colonnes libres, et $x_p = -R_{ij}$ pour le pivot de la ligne $i$, situé dans la colonne $p$. Rends ces vecteurs en **colonnes** d'un tableau de forme $(n, n - r)$ ; la forme $(n, 0)$ quand le noyau est $\{\mathbf{0}\}$ (`np.zeros((n, 0))`). Une `ValueError` si $\mathbf{A}$ n'est pas à deux dimensions.

La vérification calcule :
- a) `null_space_basis([[1, -1, 0, 2], [0, 0, 1, -3]])` (une matrice) ;
- b) `null_space_basis([[2, 4], [1, 2], [3, 6]])` (une matrice) ;
- c) la forme de `null_space_basis(np.eye(3))` (une liste de 2 entiers) ;
- d) `null_space_basis(np.zeros((2, 3)))` (une matrice) ;
- e) la forme de `N = null_space_basis(A)` pour une matrice `A` de forme $(5, 8)$ tirée au hasard (`np.random.default_rng(27)`, loi normale) ;
- f) vrai ou faux : pour ce même `A`, tous les nombres de `A @ N` sont inférieurs à $10^{-10}$ en valeur absolue ;
- g) le nom de l'erreur levée par `null_space_basis([1, 2, 3])`.

Dans ta copie : lis a) comme deux relations entre les colonnes de la matrice. Pourquoi c) a-t-elle zéro colonne, et d) trois ? Que dit le théorème du rang pour e) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")  # reload your saved file

In [ ]:
A_27 = np.random.default_rng(27).normal(size=(5, 8))
with checked("M12a.27a"):
    wbm.check("M12a.27a", np.asarray(mymath.linmaps.null_space_basis([[1, -1, 0, 2], [0, 0, 1, -3]])).tolist(),
              computed=True)
with checked("M12a.27b"):
    wbm.check("M12a.27b", np.asarray(mymath.linmaps.null_space_basis([[2, 4], [1, 2], [3, 6]])).tolist(),
              computed=True)
with checked("M12a.27c"):
    wbm.check("M12a.27c", list(np.shape(mymath.linmaps.null_space_basis(np.eye(3)))), computed=True)
with checked("M12a.27d"):
    wbm.check("M12a.27d", np.asarray(mymath.linmaps.null_space_basis(np.zeros((2, 3)))).tolist(), computed=True)
with checked("M12a.27e"):
    wbm.check("M12a.27e", list(np.shape(mymath.linmaps.null_space_basis(A_27))), computed=True)
with checked("M12a.27f"):
    wbm.check("M12a.27f", bool(np.all(np.abs(A_27 @ np.asarray(mymath.linmaps.null_space_basis(A_27))) < 1e-10)),
              computed=True)
with checked("M12a.27g"):
    wbm.check("M12a.27g", raised(mymath.linmaps.null_space_basis, [1, 2, 3]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12a_linmaps.py", "test_null_space_basis_ or test_linmaps_uses_")

### Ex M12a.28 — Coordonnées dans une base et changement de base · `coordinates, change_of_basis` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le calcul des coordonnées dans une base et la matrice d'une application dans de nouvelles bases.  
**Prérequis :** fiche §3 et §7 ; Ex M12a.26 · **mymath :** `linmaps.py` · **Parcours :** C

Écris, en suivant les docstrings :
- `coordinates(B, v, tol=1e-10)` (fiche §3) : les coordonnées $\mathbf{c}$ de $\mathbf{v}$ dans la base formée par les colonnes de $\mathbf{B}$. Vérifie les formes, puis que les colonnes sont libres (`is_independent`), résous $\mathbf{B}^\top\mathbf{B}\,\mathbf{c} = \mathbf{B}^\top\mathbf{v}$ (`np.linalg.solve`), et vérifie que $\mathbf{B}\mathbf{c}$ redonne $\mathbf{v}$, à $\mathrm{tol} \times \max(1, \lVert \mathbf{v} \rVert)$ près : sinon $\mathbf{v}$ n'est pas dans le sous-espace engendré, et c'est une `ValueError` ;
- `change_of_basis(A, S, T=None)` (fiche §7) : la matrice $\mathbf{T}^{-1}\mathbf{A}\mathbf{S}$ ($\mathbf{T} = \mathbf{S}$ par défaut), calculée en résolvant $\mathbf{T}\mathbf{M} = \mathbf{A}\mathbf{S}$, jamais avec une inverse. Une `ValueError` si les formes ne vont pas, ou si $\mathbf{S}$ ou $\mathbf{T}$ est singulière, **même** quand $\mathbf{T}$ est donnée et inversible (`np.linalg.solve(T, A @ S)` ne voit pas une $\mathbf{S}$ singulière : teste-la).

La vérification calcule :
- a) `coordinates([[1, 2], [1, 3]], [5, 7])` (une liste, 6 décimales) ;
- b) `coordinates([[1, 0], [2, 1], [0, 3]], [3, 8, 6])` (une liste, 6 décimales) ;
- c) le nom de l'erreur levée par `coordinates([[1, 0], [2, 1], [0, 3]], [1, 1, 1])` ;
- d) `change_of_basis([[4, 1], [0, -1]], [[1, 1], [0, -5]])` (une matrice, 6 décimales) ;
- e) `change_of_basis([[1, 2, 0], [0, 1, 1]], np.eye(3), [[1, 1], [1, -1]])` (une matrice, 6 décimales) ;
- f) le nom de l'erreur levée par `change_of_basis([[1, 0], [0, 1]], [[4, 2], [-6, -3]])`.

Dans ta copie : vérifie a) en recalculant la combinaison des colonnes. Pourquoi c) lève-t-elle une erreur ? Que remarques-tu en d) sur les colonnes de $\mathbf{S}$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")  # reload your saved file

In [ ]:
with checked("M12a.28a"):
    wbm.check("M12a.28a", np.asarray(mymath.linmaps.coordinates([[1, 2], [1, 3]], [5, 7])).tolist(), computed=True)
with checked("M12a.28b"):
    wbm.check("M12a.28b", np.asarray(mymath.linmaps.coordinates([[1, 0], [2, 1], [0, 3]], [3, 8, 6])).tolist(),
              computed=True)
with checked("M12a.28c"):
    wbm.check("M12a.28c", raised(mymath.linmaps.coordinates, [[1, 0], [2, 1], [0, 3]], [1, 1, 1]), computed=True)
with checked("M12a.28d"):
    wbm.check("M12a.28d", np.asarray(mymath.linmaps.change_of_basis([[4, 1], [0, -1]], [[1, 1], [0, -5]])).tolist(),
              computed=True)
with checked("M12a.28e"):
    wbm.check("M12a.28e", np.asarray(mymath.linmaps.change_of_basis([[1, 2, 0], [0, 1, 1]], np.eye(3),
                                                                    [[1, 1], [1, -1]])).tolist(), computed=True)
with checked("M12a.28f"):
    wbm.check("M12a.28f", raised(mymath.linmaps.change_of_basis, [[1, 0], [0, 1]], [[4, 2], [-6, -3]]),
              computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12a_linmaps.py", "test_coordinates_ or test_change_of_basis_ or test_linmaps_uses_")

### Ex M12a.29 — `sympy.Matrix.nullspace`, `columnspace` et `rank` contre tes fonctions 📦 ★ ⏱️ 15 min
**Objectif :** faire les mêmes calculs avec SymPy (exact) et NumPy (flottant), lire ce que rend chaque fonction, et comparer avec ta librairie.  
**Prérequis :** fiche §4 et §6 ; Ex M12a.26 et M12a.27 · **Parcours :** C, E

En pratique, on ne réécrit pas ces fonctions :
- `sympy.Matrix(A).rank()`, `.nullspace()` et `.columnspace()` calculent **exactement** (avec des fractions) le rang, une base du noyau (une liste de vecteurs colonnes, un par colonne libre, avec la même convention que `null_space_basis`) et une base de l'espace des colonnes (les colonnes à pivot) ;
- `np.linalg.matrix_rank(A)` calcule le rang en virgule flottante, par une autre méthode (la SVD, M12d) ; `scipy.linalg.null_space(A)` rend une base du noyau, mais **orthonormée** (M12b), donc d'autres vecteurs que ceux de SymPy.

Avec $\mathbf{P}$ = `[[1, 3, 0, 2], [2, 6, 1, 3], [-1, -3, 2, -4]]`, calcule :
- a) `sympy.Matrix(P).rank()` ;
- b) le nombre de vecteurs que rend `sympy.Matrix(P).nullspace()` ;
- c) le premier de ces vecteurs, en liste (`list(...)`) ;
- d) les vecteurs de `sympy.Matrix(P).columnspace()`, côte à côte (une matrice : `sympy.Matrix.hstack(*...)`) ;
- e) vrai ou faux : ta fonction `null_space_basis(P)` donne les mêmes vecteurs que SymPy, à $10^{-9}$ près (calcule-le) ;
- f) `np.linalg.matrix_rank` de la transposée de $\mathbf{P}$.

Dans ta copie : quelles relations entre les colonnes de $\mathbf{P}$ les vecteurs du noyau expriment-ils ? Pourquoi f) donne-t-il le même nombre que a) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")
answer_M12a_29a = ...  # a) an integer
answer_M12a_29b = ...  # b) an integer
answer_M12a_29c = ...  # c) a list: list(...)
answer_M12a_29d = ...  # d) a matrix
answer_M12a_29f = ...  # f) an integer
answer_M12a_29e = None
with wbm.attempt("M12a.29e"):        # your null_space_basis is called here: ⏳ while it is not written
    answer_M12a_29e = ...  # e) True or False: np.allclose(mymath.linmaps.null_space_basis(...), SymPy's basis)

In [ ]:
for letter, answer in zip("abcdef", [answer_M12a_29a, answer_M12a_29b, answer_M12a_29c, answer_M12a_29d,
                                     answer_M12a_29e, answer_M12a_29f]):
    wbm.check(f"M12a.29{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §4 à §6. Le rang de matrices construites, le rang numérique et la tolérance, une application du plan qui écrase une direction.

### Ex M12a.30 — Prédire le rang : matrices aléatoires, produits, colonnes répétées 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir le rang de matrices construites de différentes façons, puis le vérifier.  
**Prérequis :** fiche §4 · **Parcours :** C, E

Pour chaque matrice, choisis le numéro de la bonne réponse :
- a) une matrice de forme $(4, 6)$ dont les nombres sont tirés au hasard (loi normale) : (1) rang 4 ; (2) rang 6 ; (3) rang inférieur à 4 ;
- b) le produit d'une matrice aléatoire $(5, 2)$ par une matrice aléatoire $(2, 5)$, de forme $(5, 5)$ : (1) rang 5 ; (2) rang 2 ; (3) rang 4 ;
- c) une matrice aléatoire $(4, 3)$, à laquelle on ajoute une quatrième colonne égale à la somme des deux premières : (1) rang 3 ; (2) rang 4 ; (3) rang 2 ;
- d) une matrice aléatoire $(3, 5)$ (3 exemples, 5 features), centrée colonne par colonne : (1) rang 1 ; (2) rang 3 ; (3) rang 2 ;
- e) la matrice $(6, 6)$ de terme général $m_{ij} = i + j$ : (1) rang 6 ; (2) rang 1 ; (3) rang 2.

**Avant d'exécuter quoi que ce soit**, écris tes cinq prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M12a_30a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M12a_30b = ...  # b) 1, 2 or 3
prediction_M12a_30c = ...  # c) 1, 2 or 3
prediction_M12a_30d = ...  # d) 1, 2 or 3
prediction_M12a_30e = ...  # e) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcde", [prediction_M12a_30a, prediction_M12a_30b, prediction_M12a_30c,
                                       prediction_M12a_30d, prediction_M12a_30e]):
    wbm.check(f"M12a.30{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes cinq prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M12a_30a, prediction_M12a_30b, prediction_M12a_30c, prediction_M12a_30d, prediction_M12a_30e]):
    print("⏳ Écris d'abord tes cinq prédictions, puis exécute à nouveau.")
else:
    rng_30 = np.random.default_rng(30)
    M_a = rng_30.normal(size=(4, 6))
    M_b = rng_30.normal(size=(5, 2)) @ rng_30.normal(size=(2, 5))
    base_c = rng_30.normal(size=(4, 3))
    M_c = np.column_stack([base_c, base_c[:, 0] + base_c[:, 1]])
    base_d = rng_30.normal(size=(3, 5))
    M_d = base_d - base_d.mean(axis=0)
    i_e, j_e = np.indices((6, 6))
    M_e = (i_e + j_e).astype(float)
    for label, M in [("a) normale (4, 6)", M_a), ("b) (5, 2) @ (2, 5)", M_b), ("c) (4, 3) + une somme", M_c),
                     ("d) (3, 5) centrée", M_d), ("e) M[i, j] = i + j", M_e)]:
        print(f"{label:24s} forme {M.shape} → rang {np.linalg.matrix_rank(M)}")

**Après l'expérience**, dans ta copie : pourquoi une matrice tirée au hasard est-elle (presque sûrement) de rang plein ? Quelle règle du cours explique b) ? En e), écris chaque ligne comme une combinaison de deux vecteurs fixes.

### Ex M12a.31 — Le rang numérique : une colonne presque redondante et le choix de la tolérance 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer à partir de quand une relation presque exacte est vue comme exacte, par le pivot de Gauss et par NumPy, et comprendre le rôle de la tolérance et de l'échelle.  
**Prérequis :** fiche §4 ; Ex M12a.26 · **Parcours :** C

La fonction fournie `make_almost_redundant(k, scale)` fabrique un tableau de 50 exemples et 3 features dont la troisième colonne vaut la somme des deux premières **plus** $10^{-k}$ fois un bruit, le tout multiplié par `scale`. Pour $k$ petit, les trois colonnes sont libres ; pour $k$ grand, la relation est exacte à la précision des flottants près. Écris :
- `rank_table(scale=1.0, tol=1e-10)` : pour $k = 1, \ldots, 16$ (la liste `KS_31`), deux listes de 16 entiers : le rang du tableau donné par `np.linalg.matrix_rank`, et le nombre de colonnes de `mymath.linmaps.column_space_basis(..., tol=tol)` (ta fonction de M12a.26, avec la tolérance absolue `tol`).

La cellule suivante affiche les deux listes pour trois réglages : l'échelle 1 avec la tolérance $10^{-10}$, l'échelle $10^6$ avec la même tolérance, et l'échelle 1 avec la tolérance $10^{-6}$. Lis-les, puis réponds :
- a) le plus petit $k$ pour lequel `matrix_rank` donne 2, à l'échelle 1 ;
- b) le plus petit $k$ pour lequel tes pivots (tolérance $10^{-10}$) donnent 2, à l'échelle 1 ;
- c) le rang donné par tes pivots pour $k = 16$, à l'échelle $10^6$ (tolérance $10^{-10}$) ;
- d) le rang donné par `matrix_rank` pour $k = 16$, à l'échelle $10^6$ ;
- e) le plus petit $k$ pour lequel tes pivots donnent 2 avec la tolérance $10^{-6}$, à l'échelle 1.

Dans ta copie : lequel des deux rangs est « le bon » pour $k = 12$ ? Pourquoi multiplier tout le tableau par $10^6$ change-t-il le rang de tes pivots, mais pas celui de `matrix_rank` (relis la fiche, §4) ? Des features mesurées avec 4 chiffres significatifs : quelle tolérance choisirais-tu, et sur quel tableau (pense à mettre les colonnes à la même échelle) ?

In [ ]:
def make_almost_redundant(k, scale=1.0):
    """A (50, 3) table whose third column is the sum of the first two plus 10**(-k) times some noise, all times scale."""
    rng = np.random.default_rng(31)
    X = rng.normal(size=(50, 2))
    noise = rng.normal(size=50)
    return scale * np.column_stack([X, X[:, 0] + X[:, 1] + 10.0 ** (-k) * noise])


KS_31 = list(range(1, 17))

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")


def rank_table(scale=1.0, tol=1e-10):
    """For k in KS_31: the ranks of make_almost_redundant(k, scale) by np.linalg.matrix_rank and by your pivots."""
    raise NotImplementedError("deux listes : np.linalg.matrix_rank, puis column_space_basis(..., tol=tol).shape[1]")

In [ ]:
with checked("M12a.31 tableaux"):
    for scale_t, tol_t in [(1.0, 1e-10), (1e6, 1e-10), (1.0, 1e-6)]:
        numpy_t, pivots_t = rank_table(scale=scale_t, tol=tol_t)
        print(f"échelle {scale_t:g}, tol {tol_t:g}")
        print("   k          :", " ".join(f"{k:2d}" for k in KS_31))
        print("   matrix_rank:", " ".join(f"{r:2d}" for r in numpy_t))
        print("   pivots     :", " ".join(f"{r:2d}" for r in pivots_t))

answer_M12a_31a = ...  # a) an integer, read in the tables above
answer_M12a_31b = ...  # b) an integer
answer_M12a_31c = ...  # c) an integer
answer_M12a_31d = ...  # d) an integer
answer_M12a_31e = ...  # e) an integer
for letter, answer in zip("abcde", [answer_M12a_31a, answer_M12a_31b, answer_M12a_31c, answer_M12a_31d,
                                    answer_M12a_31e]):
    wbm.check(f"M12a.31{letter}", answer)

### Ex M12a.32 — Une application linéaire du plan : l'image du carré unité, et ce qu'elle écrase quand son noyau n'est pas nul 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner l'image du carré unité par une application linéaire du plan, et, quand elle n'est pas inversible, son noyau, son image et les droites qu'elle écrase en un point.  
**Prérequis :** fiche §5 et §6 ; M09a.33 · **Parcours :** C, E

Écris `draw_kernel_and_image(ax, M)`, qui trace dans l'axe `ax`, pour une matrice $\mathbf{M}$ de forme $(2, 2)$ :
- l'image du carré unité, de sommets $(0, 0)$, $(1, 0)$, $(1, 1)$, $(0, 1)$ **dans cet ordre**, avec `matplotlib.patches.Polygon` et `ax.add_patch` (le premier polygone de l'axe) ;
- si $\mathbf{M}$ n'est **pas** inversible (rang 1) : le noyau, une droite par l'origine de direction $\mathbf{k}$ (`mymath.linmaps.null_space_basis`, ou à la main), tracée pour $t$ entre $-2$ et $2$ avec `label="noyau"` ; l'image, la droite par l'origine portée par une colonne non nulle de $\mathbf{M}$, tracée avec `label="image"` ; et, pour les trois sommets $(1, 0)$, $(0, 1)$, $(1, 1)$ du carré, le segment $\mathbf{p} + t\,\mathbf{k}$, $t \in [-1, 1]$, en pointillés, avec un point en $\mathbf{M}\mathbf{p}$ ;
- si $\mathbf{M}$ est inversible : aucune ligne de label « noyau ».

La cellule de la figure s'en sert pour une matrice de rang 1 et pour une matrice inversible ; une cellule de vérification essaie ta fonction sur trois matrices, dans des axes neufs. Pour savoir si $\mathbf{M}$ est inversible, regarde la forme de la base du noyau.

Dans ta copie : que devient le carré unité par la matrice de rang 1 ? Pourquoi chaque segment en pointillés est-il envoyé sur un seul point ? Où sont ces points ? Pourquoi aucune matrice ne peut-elle défaire la première transformation (fiche §8) ?

In [ ]:
from matplotlib.patches import Polygon

mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")


def draw_kernel_and_image(ax, M):
    """In ax: the image of the unit square (a Polygon), and for a singular M its kernel, its image and three segments."""
    raise NotImplementedError("le carré : Polygon ; puis, si le noyau n'est pas {0}, les lignes « noyau » et « image »")

In [ ]:
with checked("M12a.32 figure"):
    matrices_32 = [("rang 1", [[1, -2], [-0.5, 1]]), ("inversible", [[1.5, 0.5], [-0.5, 1]])]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
    try:
        for ax, (name, M) in zip(axes, matrices_32):
            draw_kernel_and_image(ax, M)
            ax.set(title=f"{name} : M = {M}", aspect="equal", xlim=(-3, 3), ylim=(-3, 3))
            if ax.get_legend_handles_labels()[0]:
                ax.legend(loc="lower right", fontsize=8)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

**Vérifications automatiques** (sur trois matrices, dans des axes neufs).

In [ ]:
with checked("M12a.32 vérifications"):
    problems = []
    for M_t, singular_t in [([[1, -2], [-0.5, 1]], True), ([[1.5, 0.5], [-0.5, 1]], False), ([[0, 0], [3, 1]], True)]:
        fig_t, ax_t = plt.subplots()
        try:
            created = drawn(lambda: draw_kernel_and_image(ax_t, M_t))
            labels_t = {line.get_label(): line for line in ax_t.lines}
            polygons_t = [p for p in ax_t.patches if hasattr(p, "get_xy")]
        finally:
            plt.close(fig_t)
        M_arr = np.array(M_t, dtype=float)
        if created:
            problems.append("draw_kernel_and_image ouvre sa propre figure : dessine dans l'axe reçu")
            break
        corners = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=float)
        if not polygons_t or not np.allclose(np.asarray(polygons_t[0].get_xy(), dtype=float)[:4], corners @ M_arr.T,
                                             atol=1e-9):
            problems.append(f"pour M = {M_t}, le premier polygone n'a pas pour sommets les images de (0, 0), (1, 0), "
                            "(1, 1), (0, 1), dans cet ordre")
            break
        if singular_t:
            if "noyau" not in labels_t or "image" not in labels_t:
                problems.append(f"pour M = {M_t}, il manque une ligne de label « noyau » ou « image »")
                break
            ker_pts = np.column_stack([labels_t["noyau"].get_xdata(), labels_t["noyau"].get_ydata()]).astype(float)
            img_pts = np.column_stack([labels_t["image"].get_xdata(), labels_t["image"].get_ydata()]).astype(float)
            if len(ker_pts) < 2 or np.ptp(ker_pts, axis=0).max() < 1e-9 or not np.allclose(ker_pts @ M_arr.T, 0, atol=1e-9):
                problems.append(f"pour M = {M_t}, les points de la ligne « noyau » ne vérifient pas M x = 0 (ou la "
                                "ligne est réduite à un point)")
                break
            column = M_arr[:, np.argmax(np.abs(M_arr).sum(axis=0))]
            cross = img_pts[:, 0] * column[1] - img_pts[:, 1] * column[0]
            if len(img_pts) < 2 or np.ptp(img_pts, axis=0).max() < 1e-9 or not np.allclose(cross, 0, atol=1e-9):
                problems.append(f"pour M = {M_t}, la ligne « image » n'est pas portée par une colonne de M")
                break
        elif "noyau" in labels_t:
            problems.append(f"pour M = {M_t}, inversible, le noyau est {{0}} : pas de ligne « noyau »")
            break
    if problems:
        print("❌ Ex M12a.32 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M12a.32 : le carré, le noyau et l'image sont justes, pour trois matrices.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §2, §6, §7 et §8. Trois erreurs à trouver, un test de propriétés, puis des relations cachées à retrouver sans en inventer.

### Ex M12a.33 — Chasse à l'erreur : une « base » liée, un noyau faux, S⁻¹AS écrit à l'envers 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur les bases, les noyaux et les changements de base, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §2, §6, §7 et §8 · **Parcours :** C

Un collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle doit calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
A = np.array([[1.0, 2.0, 1.0], [2.0, 4.0, 3.0]])                     # a 2 × 3 matrix
u, v = np.array([1.0, 0.0, 1.0]), np.array([0.0, 1.0, 1.0])
B = np.column_stack([u, v, u + v])           # (A) a basis of R^3 that completes (u, v): its rank
R = np.array([[1.0, 2.0, 0.0], [0.0, 0.0, 1.0]])                     # the reduced form of A
k = np.array([R[0, 1], 1.0, 0.0])            # (B) the basis vector of ker A with x2 = 1: k[0]
e3 = np.array([0.0, 0.0, 1.0])
c = np.linalg.solve(np.column_stack([u, v, e3]), np.array([2.0, 3.0, 4.0]))   # (C) coordinates of (2, 3, 4) in (u, v, e3): c[2]
M = np.array([[2.0, 1.0], [0.0, 3.0]])
S = np.array([[1.0, 1.0], [0.0, 1.0]])
D = S @ M @ np.linalg.inv(S)                 # (D) the matrix of x -> M x in the basis of the columns of S: D[0, 1]
n = A.shape[1] - 2                           # (E) dim ker A, by the rank theorem (A has rank 2)
y = np.linalg.solve(M, np.array([5.0, 6.0]) - np.array([1.0, 0.0]))         # (F) phi^-1(5, 6) for phi(x) = M x + (1, 0): y[0]
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Laquelle des trois s'exécute sans erreur **et** sans valeur absurde, et pourquoi est-elle la plus dangereuse ?

In [ ]:
answer_M12a_33a = ...  # a) letters, as text
answer_M12a_33b = ...  # b) a list of 6 values

In [ ]:
wbm.check("M12a.33a", answer_M12a_33a)
wbm.check("M12a.33b", answer_M12a_33b)

### Ex M12a.34 — Un test de propriété : A fois une base du noyau donne 0, et le théorème du rang sur des matrices aléatoires 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test fondé sur des propriétés vérifiées sur des matrices aléatoires de rang choisi, et voir quelle propriété attrape quelle erreur.  
**Prérequis :** fiche §6 ; Ex M12a.26 et M12a.27 · **Parcours :** C

Les tests de `mymath` comparent tes fonctions à SymPy. On peut aussi vérifier des **propriétés** que tout résultat juste doit avoir. Écris :
- `check_kernel_and_rank(null_space_basis, column_space_basis, seed, trials=10)` : avec `rng = np.random.default_rng(seed)`, répète `trials` fois : tire $m$ et $n$ entre 1 et 6 (`rng.integers(1, 7, size=2)`), un rang $r$ entre 0 et $\min(m, n)$ (`rng.integers(0, min(m, n) + 1)`), et la matrice entière $\mathbf{A} = \mathbf{L}\mathbf{C}$, produit de deux matrices d'entiers tirés entre $-3$ et 3, de formes $(m, r)$ et $(r, n)$ (`rng.integers(-3, 4, size=...)`, converties en floats). Avec `N = null_space_basis(A)` et `Q = column_space_basis(A)`, vérifie par des `assert` (avec un message) : (1) $\mathbf{A}\mathbf{N} = \mathbf{0}$ à $10^{-9}$ près ; (2) le théorème du rang, `N.shape[1] + Q.shape[1] == n` ; (3) les colonnes de `N` sont libres (`np.linalg.matrix_rank(N) == N.shape[1]`, permis ici : c'est le test, pas `linmaps.py` ; `matrix_rank` refuse un tableau sans colonne, et un noyau $\{\mathbf{0}\}$ n'a rien à vérifier) ;
- `test_kernel_and_rank()` : le test au format pytest, sans argument, qui appelle `check_kernel_and_rank` avec les fonctions de **ta** librairie (`mymath.linmaps`) et une graine fixe.

La cellule fournie définit deux copies fausses : un noyau qui a la bonne forme mais ne contient que des zéros, et une « base de l'image » qui garde toutes les colonnes. La vérification lance ton test sur les bonnes fonctions (la référence) et sur chaque copie fausse, pour 20 graines.

Dans ta copie : quelle propriété attrape le noyau fait de zéros, et pourquoi les deux autres le laissent-elles passer ? Quelle propriété attrape l'image faite de toutes les colonnes ? Une fonction fausse pourrait-elle passer les trois propriétés ? (Pense à un noyau juste, mais rendu avec ses colonnes multipliées par 2.)

In [ ]:
def null_space_zeros(A, tol=1e-10):          # a copy with a bug: the right shape, but only zeros
    A = np.asarray(A, dtype=float)
    rank = np.linalg.matrix_rank(A)
    return np.zeros((A.shape[1], A.shape[1] - rank))


def column_space_all(A, tol=1e-10):          # a copy with another bug: every column, pivot or not
    return np.array(A, dtype=float)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")


def check_kernel_and_rank(null_space_basis, column_space_basis, seed, trials=10):
    """Assert A N = 0, the rank theorem and independent kernel columns, on `trials` random integer matrices."""
    raise NotImplementedError("tire m, n, r, puis A = L C ; trois assert, chacun avec un message")


def test_kernel_and_rank():
    """The pytest test: check_kernel_and_rank on your null_space_basis and column_space_basis, with a fixed seed."""
    raise NotImplementedError("un appel à check_kernel_and_rank")

In [ ]:
with checked("M12a.34 vérifications"):
    problems = []
    ref_34 = wbm.load_mymath("ref", alias="mymath_ref").linmaps   # your mymath stays loaded

    def outcome(null_space_t, column_space_t, seed):
        """None if check_kernel_and_rank passes, else the message of its AssertionError."""
        try:
            check_kernel_and_rank(null_space_t, column_space_t, seed)
        except AssertionError as exc:
            return str(exc)
        return None

    try:
        passed = sum(outcome(ref_34.null_space_basis, ref_34.column_space_basis, seed) is None for seed in range(20))
        caught_zeros = [outcome(null_space_zeros, ref_34.column_space_basis, seed) for seed in range(20)]
        caught_all = [outcome(ref_34.null_space_basis, column_space_all, seed) for seed in range(20)]
    except ValueError as exc:
        if "zero-size" not in str(exc):
            raise
        print("❌ Ex M12a.34 : np.linalg.matrix_rank refuse un tableau sans colonne : quand le noyau est {0} "
              "(N.shape[1] == 0), saute la vérification d'indépendance.")
        raise StopCheck from None
    print(f"la bonne paire passe ton test {passed} fois sur 20 ; le noyau fait de zéros est attrapé "
          f"{sum(m is not None for m in caught_zeros)} fois sur 20 ; l'image faite de toutes les colonnes, "
          f"{sum(m is not None for m in caught_all)} fois sur 20")
    if passed < 20:
        problems.append("ton test échoue sur les bonnes fonctions : compare avec np.allclose (des flottants), et "
                        "vérifie tes formes")
    if sum(m is not None for m in caught_zeros) < 20:
        problems.append("ton test laisse passer un noyau fait de zéros : vérifie aussi que ses colonnes sont libres")
    if sum(m is not None for m in caught_all) < 20:
        problems.append("ton test laisse passer une image faite de toutes les colonnes : vérifie le théorème du rang, "
                        "nombre de colonnes de la base de l'image + nombre de colonnes de la base du noyau = n")
    if not problems and any(not (m or "").strip() for m in caught_zeros + caught_all):
        problems.append("ajoute un message à chaque assert (assert condition, message)")
    if not problems:
        try:
            test_kernel_and_rank()
        except AssertionError as exc:
            problems.append(f"test_kernel_and_rank échoue sur ta librairie : {exc}")
    if problems:
        print("❌ Ex M12a.34 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M12a.34 : ton test passe sur les bonnes fonctions et attrape les deux copies fausses.")

### Ex M12a.35 — Défi : retrouver les relations linéaires exactes cachées entre les colonnes d'un tableau, sans en inventer 🏆 ★★★ ⏱️ 30 min
**Objectif :** trouver toutes les relations linéaires exactes entre les colonnes d'un tableau de données, quelles que soient les échelles des colonnes, sans prendre une relation approchée pour une relation exacte.  
**Prérequis :** fiche §4 et §6 ; Ex M12a.27 et M12a.31 · **Parcours :** C

La fonction fournie `make_hidden_table(seed)` fabrique un tableau de 150 exemples dont certaines colonnes sont des combinaisons **exactes** des autres (aux arrondis des flottants près), et une colonne qui n'est une combinaison des autres qu'**à $10^{-6}$ près** en relatif. Les colonnes ont des ordres de grandeur très différents (de $10^{-4}$ à $10^{6}$), et sont mélangées.

**L'objectif** : écrire `hidden_relations(X)`, qui rend une matrice de forme $(p, k)$ dont les colonnes forment une base de l'espace des relations **exactes** $\{\mathbf{c} : \mathbf{X}\mathbf{c} = \mathbf{0}\}$ : toutes les relations exactes, et aucune de trop (la relation à $10^{-6}$ près n'en est pas une). La vérification essaie ta fonction sur trois tableaux (`seed` = 35, 36 et 37) ; une dernière cellule enregistre si l'objectif est atteint sur les trois.

`null_space_basis(X)` avec sa tolérance absolue par défaut échoue : les arrondis d'une colonne de l'ordre de $10^6$ dépassent $10^{-10}$, et le bruit d'une colonne de l'ordre de $10^{-4}$ passe sous $10^{-10}$ (M12a.31). Pistes : mettre les colonnes à la même échelle (diviser chaque colonne par sa norme, une matrice diagonale $\mathbf{D}$ : $\mathbf{X}\mathbf{D}^{-1}$), chercher le noyau de ce tableau avec une tolérance bien choisie entre les arrondis (vers $10^{-15}$) et le bruit (vers $10^{-6}$), puis revenir aux relations du tableau de départ (si $\mathbf{X}\mathbf{D}^{-1}\mathbf{y} = \mathbf{0}$, alors $\mathbf{c} = \mathbf{D}^{-1}\mathbf{y}$ vérifie $\mathbf{X}\mathbf{c} = \mathbf{0}$).

Dans ta copie : quelle tolérance as-tu choisie, et pourquoi ? Que se passe-t-il avec une tolérance de $10^{-5}$, et avec la tolérance par défaut sans changer d'échelle ? Sur de vraies données mesurées, comment choisirais-tu la tolérance ?

In [ ]:
def make_hidden_table(seed, n=150):
    """A (n, p) table whose columns hide exact linear relations, and one relation that is only almost exact."""
    rng = np.random.default_rng(seed)
    q, k = int(rng.integers(3, 5)), int(rng.integers(1, 4))      # q free directions, k exact relations
    G = np.zeros((q, q + k + 1))
    G[:, :q] = np.eye(q)
    for j in range(q, q + k + 1):
        column = np.zeros(q)
        while np.count_nonzero(column) < 2:
            column = rng.integers(-3, 4, size=q).astype(float)
        G[:, j] = column
    scales = 10.0 ** rng.integers(-2, 4, size=q + k + 1)
    scales[q], scales[q + k] = 1e6, 1e-4
    X = (rng.normal(size=(n, q)) @ G) * scales
    X[:, q + k] += 1e-6 * np.linalg.norm(X[:, q + k]) / np.sqrt(n) * rng.normal(size=n)   # almost, not exactly
    return X[:, rng.permutation(q + k + 1)]

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12a")


def hidden_relations(X):
    """A (p, k) matrix whose columns are a basis of the exact relations X c = 0 between the columns of X."""
    raise NotImplementedError("mettre les colonnes à la même échelle, un noyau avec une tolérance choisie, revenir")

In [ ]:
with checked("M12a.35 défi"):
    problems, ok_35 = [], False
    for seed_t in (35, 36, 37):
        X_t = make_hidden_table(seed_t)
        rng_t = np.random.default_rng(seed_t)
        rng_t.integers(3, 5)
        k_true = int(rng_t.integers(1, 4))
        try:
            N_t = np.asarray(hidden_relations(X_t.copy()), dtype=float)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"hidden_relations lève {type(exc).__name__} ({exc})")
            break
        p_t = X_t.shape[1]
        if N_t.ndim != 2 or N_t.shape[0] != p_t:
            problems.append(f"table {seed_t} : hidden_relations rend la forme {N_t.shape}, pas ({p_t}, k)")
            break
        norms_t = np.linalg.norm(X_t, axis=0)
        scaled_t = N_t * norms_t[:, None]                  # the relations between the normalized columns
        if N_t.shape[1] < k_true:
            problems.append(f"table {seed_t} : {N_t.shape[1]} relation(s) trouvée(s), il en manque : une colonne d'une "
                            "échelle très différente des autres cache peut-être une relation exacte")
            break
        if N_t.shape[1] > k_true:
            problems.append(f"table {seed_t} : {N_t.shape[1]} relations, c'est trop : une relation vraie à 1e-6 près "
                            "seulement n'est pas exacte ; ta tolérance est trop large, ou elle dépend de l'échelle")
            break
        if k_true:
            col_norms_t = np.linalg.norm(scaled_t, axis=0)
            residual_t = np.linalg.norm((X_t / norms_t) @ scaled_t, axis=0) / col_norms_t
            if np.any(col_norms_t == 0) or residual_t.max() > 1e-8:
                problems.append(f"table {seed_t} : une relation rendue n'est pas exacte (résidu relatif "
                                f"{residual_t.max():.1e}) ; si tu as changé d'échelle, reviens aux relations de X : "
                                "c = D⁻¹y, chaque coordonnée divisée par la norme de sa colonne")
                break
            if np.linalg.matrix_rank(scaled_t / col_norms_t, tol=1e-6) < k_true:
                problems.append(f"table {seed_t} : les relations rendues ne sont pas libres")
                break
        print(f"table {seed_t} : {X_t.shape}, {N_t.shape[1]} relation(s) exacte(s) retrouvée(s)")
    if problems:
        print("❌ Ex M12a.35 : " + " ; ".join(problems) + ".")
    else:
        ok_35 = True
        print("✅ Ex M12a.35 : défi réussi : toutes les relations exactes, et aucune de trop, dans les trois tables.")
with checked("M12a.35"):
    hidden_relations(make_hidden_table(35))                   # ⏳ while hidden_relations is not written
    wbm.check("M12a.35", bool(ok_35), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu reconnaître un sous-espace, et décider si une famille est libre ?
2. Sais-tu extraire une base, et calculer des coordonnées dans une base ?
3. Sais-tu calculer un rang, et majorer celui d'un produit ?
4. Sais-tu écrire la matrice d'une application linéaire, et calculer un noyau et une image ?
5. Sais-tu changer de base, et inverser une application affine ?

**La suite** : le chapitre 12 de workbookIA (préparation des données), en parallèle de M12b, l'orthogonalité et les projections.